# Sistemas Inteligentes I
## Búsqueda adversarial: algoritmo Minimax

**Autor:** Jairo I. Vélez B.

---

# 1. Idea central: buscar cuando existe un adversario

En BFS, DFS o A* buscamos una ruta hacia un objetivo.  
En un juego ocurre algo diferente:

> después de que nosotros elegimos una acción, **otro jugador también elige**.

Por tanto, no basta con preguntarnos:

> ¿Cuál es la mejor jugada que puedo hacer?

También debemos considerar:

> ¿Qué hará mi oponente si intenta perjudicarme?

Minimax modela dos jugadores:

- **MAX:** intenta obtener el valor más alto posible.
- **MIN:** intenta obtener el valor más bajo posible.

Supondremos inicialmente que ambos jugadores actúan de manera racional.

# 2. Elementos de un problema adversarial

Un juego puede describirse mediante:

- **Estado:** configuración actual del juego.
- **Jugador actual:** indica quién debe mover.
- **Acciones:** movimientos legales desde el estado actual.
- **Resultado:** estado que se obtiene al aplicar una acción.
- **Estado terminal:** posición en la que la partida ha terminado.
- **Utilidad:** valor numérico asociado al resultado final.

Una convención sencilla puede ser:

| Resultado para MAX | Utilidad |
|---|---:|
| Victoria | `+1` |
| Empate | `0` |
| Derrota | `-1` |

En ejemplos más generales, la utilidad puede tomar cualquier valor numérico.

# 3. Primer árbol de juego

Consideremos el siguiente árbol.

- La raíz `A` pertenece a **MAX**.
- En el siguiente nivel juega **MIN**.
- Las hojas contienen valores de utilidad.

```text
                    A  (MAX)
              /         |         \
          B (MIN)    C (MIN)    D (MIN)
          / | \       / | \       / | \
         3  5  2     9  1  4     6  7  8
```

Pregunta:

> Si MIN juega racionalmente, ¿qué opción debería escoger MAX desde `A`?

In [50]:
arbol = {
    "A": ["B", "C", "D"],
    "B": ["B1", "B2", "B3"],
    "C": ["C1", "C2", "C3"],
    "D": ["D1", "D2", "D3"],
}

utilidades = {
    "B1": 3, "B2": 5, "B3": 2,
    "C1": 9, "C2": 1, "C3": 4,
    "D1": 6, "D2": 7, "D3": 8,
}

arbol, utilidades

({'A': ['B', 'C', 'D'],
  'B': ['B1', 'B2', 'B3'],
  'C': ['C1', 'C2', 'C3'],
  'D': ['D1', 'D2', 'D3']},
 {'B1': 3,
  'B2': 5,
  'B3': 2,
  'C1': 9,
  'C2': 1,
  'C3': 4,
  'D1': 6,
  'D2': 7,
  'D3': 8})

# 4. Razonamiento antes del algoritmo

Analicemos primero cada nodo de MIN.

### Nodo B

MIN puede elegir entre:

$$3,\ 5,\ 2$$

Por tanto:

$$\min(3,5,2)=2$$

### Nodo C

$$\min(9,1,4)=1$$

### Nodo D

$$\min(6,7,8)=6$$

La raíz pertenece a MAX, así que compara:

$$\max(2,1,6)=6$$

Por tanto, MAX debería elegir la rama `D`.

Esta idea es exactamente la que implementa **Minimax**.

# 5. Algoritmo Minimax

La definición recursiva puede escribirse como:

$$
V(s)=
\begin{cases}
U(s) & \text{si }s\text{ es terminal}\\
\max_{s' \in Sucesores(s)} V(s') & \text{si juega MAX}\\
\min_{s' \in Sucesores(s)} V(s') & \text{si juega MIN}
\end{cases}
$$

La recursión baja hasta los estados terminales y después los valores se
**propagan hacia arriba**.

In [51]:
def minimax(nodo, es_max, arbol, utilidades):
    # Caso base: nodo terminal
    if nodo in utilidades:
        return utilidades[nodo]

    valores = []

    for hijo in arbol[nodo]:
        valor = minimax(hijo, not es_max, arbol, utilidades)
        valores.append(valor)

    if es_max:
        return max(valores)
    else:
        return min(valores)


valor_raiz = minimax("A", True, arbol, utilidades)
valor_raiz

6

## 5.1 Obtener también la mejor jugada

Conocer el valor del estado es útil, pero normalmente necesitamos además saber:

> **¿qué acción debe ejecutar el jugador?**

La siguiente función devuelve el valor Minimax y el hijo seleccionado.

In [52]:
def mejor_jugada_minimax(nodo, es_max, arbol, utilidades):
    if nodo in utilidades:
        return utilidades[nodo], None

    opciones = []

    for hijo in arbol[nodo]:
        valor = minimax(hijo, not es_max, arbol, utilidades)
        opciones.append((valor, hijo))

    if es_max:
        valor, hijo = max(opciones, key=lambda x: x[0])
    else:
        valor, hijo = min(opciones, key=lambda x: x[0])

    return valor, hijo


valor, jugada = mejor_jugada_minimax("A", True, arbol, utilidades)

print("Valor Minimax:", valor)
print("Mejor jugada para MAX:", jugada)

Valor Minimax: 6
Mejor jugada para MAX: D


# 6. Minimax paso a paso

Para comprender mejor el algoritmo observaremos la recursión.

La sangría permite identificar la profundidad en el árbol.

In [53]:
def minimax_debug(nodo, es_max, arbol, utilidades, profundidad=0):
    sangria = "    " * profundidad
    jugador = "MAX" if es_max else "MIN"

    if nodo in utilidades:
        print(f"{sangria}{nodo}: terminal -> utilidad {utilidades[nodo]}")
        return utilidades[nodo]

    print(f"{sangria}{nodo}: turno de {jugador}")
    valores = []

    for hijo in arbol[nodo]:
        valor = minimax_debug(
            hijo,
            not es_max,
            arbol,
            utilidades,
            profundidad + 1
        )
        valores.append(valor)

    if es_max:
        resultado = max(valores)
    else:
        resultado = min(valores)

    print(f"{sangria}{nodo}: {jugador} selecciona {resultado}")
    return resultado


minimax_debug("A", True, arbol, utilidades)

A: turno de MAX
    B: turno de MIN
        B1: terminal -> utilidad 3
        B2: terminal -> utilidad 5
        B3: terminal -> utilidad 2
    B: MIN selecciona 2
    C: turno de MIN
        C1: terminal -> utilidad 9
        C2: terminal -> utilidad 1
        C3: terminal -> utilidad 4
    C: MIN selecciona 1
    D: turno de MIN
        D1: terminal -> utilidad 6
        D2: terminal -> utilidad 7
        D3: terminal -> utilidad 8
    D: MIN selecciona 6
A: MAX selecciona 6


6

### Preguntas de análisis

1. ¿Por qué MAX no selecciona directamente la hoja con valor `9`?
2. ¿Qué supone Minimax sobre el comportamiento del adversario?
3. ¿Qué ocurriría si MIN no escogiera siempre la opción de menor valor?
4. ¿Por qué los valores se calculan desde las hojas hacia la raíz?
5. ¿El valor de una hoja representa necesariamente una puntuación real del juego?

### Respuestas (sección 6)

1. La hoja `9` está bajo `C`, y en ese nivel elige **MIN**, que preferirá el `1`. MAX solo controla la primera decisión (`B`, `C` o `D`), no la hoja final.
2. Que el adversario es **racional y óptimo**: siempre elige el resultado que más perjudica a MAX (juego de suma cero con información perfecta).
3. Si MIN se equivoca, MAX obtiene **al menos** el valor Minimax (aquí 6) y posiblemente más. Minimax no intenta aprovechar errores: elegir `C` podría dar 9 si MIN falla, pero arriesga terminar en 1.
4. Porque los únicos valores conocidos de antemano son las utilidades de los estados terminales; el valor de un nodo interno depende de los valores de sus hijos, así que hay que evaluarlos primero y propagar hacia arriba.
5. **No necesariamente.** Es una convención que codifica preferencia (`+1`, `0`, `-1`) o, en búsqueda con profundidad limitada, una estimación heurística.

# 7. Un árbol con mayor profundidad

Ahora utilizaremos un árbol de tres decisiones.

```text
MAX → MIN → MAX → utilidad
```

Esto permite observar que los roles se alternan en cada nivel.

In [54]:
arbol_profundo = {
    "A": ["B", "C"],
    "B": ["D", "E"],
    "C": ["F", "G"],
    "D": ["D1", "D2"],
    "E": ["E1", "E2"],
    "F": ["F1", "F2"],
    "G": ["G1", "G2"],
}

utilidades_profundo = {
    "D1": 3, "D2": 5,
    "E1": 6, "E2": 9,
    "F1": 1, "F2": 2,
    "G1": 0, "G2": -1,
}

minimax_debug("A", True, arbol_profundo, utilidades_profundo)

A: turno de MAX
    B: turno de MIN
        D: turno de MAX
            D1: terminal -> utilidad 3
            D2: terminal -> utilidad 5
        D: MAX selecciona 5
        E: turno de MAX
            E1: terminal -> utilidad 6
            E2: terminal -> utilidad 9
        E: MAX selecciona 9
    B: MIN selecciona 5
    C: turno de MIN
        F: turno de MAX
            F1: terminal -> utilidad 1
            F2: terminal -> utilidad 2
        F: MAX selecciona 2
        G: turno de MAX
            G1: terminal -> utilidad 0
            G2: terminal -> utilidad -1
        G: MAX selecciona 0
    C: MIN selecciona 0
A: MAX selecciona 5


5

## 7.1 Contar nodos evaluados

En árboles pequeños Minimax resulta sencillo.  
Sin embargo, el número de posiciones posibles puede crecer rápidamente.

Contaremos cuántos nodos visita el algoritmo.

In [55]:
def minimax_contando(nodo, es_max, arbol, utilidades, contador):
    contador["visitados"] += 1

    if nodo in utilidades:
        contador["terminales"] += 1
        return utilidades[nodo]

    valores = [
        minimax_contando(hijo, not es_max, arbol, utilidades, contador)
        for hijo in arbol[nodo]
    ]

    return max(valores) if es_max else min(valores)


contador = {"visitados": 0, "terminales": 0}
valor = minimax_contando(
    "A",
    True,
    arbol_profundo,
    utilidades_profundo,
    contador
)

print("Valor Minimax:", valor)
print("Nodos visitados:", contador["visitados"])
print("Hojas evaluadas:", contador["terminales"])

Valor Minimax: 5
Nodos visitados: 15
Hojas evaluadas: 8


# 8. Caso aplicado: juego de las piedras

Trabajaremos con un juego muy sencillo:

- Existe una pila con cierta cantidad de piedras.
- En cada turno un jugador puede retirar `1`, `2` o `3` piedras.
- El jugador que retira la **última piedra gana**.

Representaremos un estado como:

```python
(piedras_restantes, jugador)
```

donde:

- `jugador = 1` representa a MAX;
- `jugador = -1` representa a MIN.

In [56]:
MOVIMIENTOS = (1, 2, 3)

def movimientos_validos(piedras):
    return [m for m in MOVIMIENTOS if m <= piedras]


for n in range(1, 8):
    print(n, "piedras ->", movimientos_validos(n))

1 piedras -> [1]
2 piedras -> [1, 2]
3 piedras -> [1, 2, 3]
4 piedras -> [1, 2, 3]
5 piedras -> [1, 2, 3]
6 piedras -> [1, 2, 3]
7 piedras -> [1, 2, 3]


## 8.1 Utilidad del estado terminal

Cuando no quedan piedras, significa que el jugador anterior tomó la última.

Si el jugador que debe mover ahora es MAX, entonces MIN realizó la jugada anterior
y ganó. Por tanto, la utilidad para MAX es `-1`.

Si debe mover MIN, MAX realizó la jugada anterior y ganó. La utilidad es `+1`.

In [57]:
def minimax_piedras(piedras, turno_max):
    if piedras == 0:
        return -1 if turno_max else 1

    valores = []

    for retirar in movimientos_validos(piedras):
        valor = minimax_piedras(
            piedras - retirar,
            not turno_max
        )
        valores.append(valor)

    return max(valores) if turno_max else min(valores)


for piedras in range(1, 11):
    print(
        f"{piedras:2d} piedras -> valor Minimax:",
        minimax_piedras(piedras, True)
    )

 1 piedras -> valor Minimax: 1
 2 piedras -> valor Minimax: 1
 3 piedras -> valor Minimax: 1
 4 piedras -> valor Minimax: -1
 5 piedras -> valor Minimax: 1
 6 piedras -> valor Minimax: 1
 7 piedras -> valor Minimax: 1
 8 piedras -> valor Minimax: -1
 9 piedras -> valor Minimax: 1
10 piedras -> valor Minimax: 1


## 8.2 Encontrar la mejor jugada

Ahora determinaremos cuántas piedras debería retirar MAX.

In [58]:
def mejor_movimiento_piedras(piedras):
    opciones = []

    for retirar in movimientos_validos(piedras):
        valor = minimax_piedras(piedras - retirar, False)
        opciones.append((valor, retirar))

    mejor_valor, mejor_movimiento = max(opciones, key=lambda x: x[0])

    return {
        "retirar": mejor_movimiento,
        "valor": mejor_valor,
        "opciones": opciones,
    }


for piedras in range(1, 11):
    print(
        f"{piedras:2d} piedras ->",
        mejor_movimiento_piedras(piedras)
    )

 1 piedras -> {'retirar': 1, 'valor': 1, 'opciones': [(1, 1)]}
 2 piedras -> {'retirar': 2, 'valor': 1, 'opciones': [(-1, 1), (1, 2)]}
 3 piedras -> {'retirar': 3, 'valor': 1, 'opciones': [(-1, 1), (-1, 2), (1, 3)]}
 4 piedras -> {'retirar': 1, 'valor': -1, 'opciones': [(-1, 1), (-1, 2), (-1, 3)]}
 5 piedras -> {'retirar': 1, 'valor': 1, 'opciones': [(1, 1), (-1, 2), (-1, 3)]}
 6 piedras -> {'retirar': 2, 'valor': 1, 'opciones': [(-1, 1), (1, 2), (-1, 3)]}
 7 piedras -> {'retirar': 3, 'valor': 1, 'opciones': [(-1, 1), (-1, 2), (1, 3)]}
 8 piedras -> {'retirar': 1, 'valor': -1, 'opciones': [(-1, 1), (-1, 2), (-1, 3)]}
 9 piedras -> {'retirar': 1, 'valor': 1, 'opciones': [(1, 1), (-1, 2), (-1, 3)]}
10 piedras -> {'retirar': 2, 'valor': 1, 'opciones': [(-1, 1), (1, 2), (-1, 3)]}


### Preguntas de análisis

1. ¿Qué cantidades iniciales de piedras representan una posición desfavorable para MAX?
2. ¿Existe algún patrón?
3. ¿Por qué algunas posiciones tienen valor `-1` incluso si MAX todavía dispone de varios movimientos?
4. ¿Puede haber más de una jugada igualmente buena?
5. ¿Qué cambiaría si fuera obligatorio retirar únicamente `1` o `2` piedras?

### Respuestas (juego de las piedras con `1, 2, 3`)

1. Las posiciones **4 y 8** (en general, los múltiplos de 4) tienen valor `-1` para MAX.
2. **Sí:** quien mueve pierde si `n` es múltiplo de 4; en los demás casos gana retirando `n mod 4` piedras, con lo que deja un múltiplo de 4 al rival.
3. Porque, retire lo que retire MAX (`k ∈ {1, 2, 3}`), MIN responde con `4 − k` y vuelve a dejar un múltiplo de 4, hasta que MIN se lleva la última piedra. Tener varios movimientos disponibles no ayuda si todos conducen a posiciones perdedoras.
4. Con `1, 2, 3` cada posición ganadora tiene **una sola** jugada ganadora (`n mod 4`). Con otras reglas sí puede haber varias: por ejemplo, con `1, 2, 4` en `n = 4, 7, 10, …` (ver 10.5).
5. Con solo `1` o `2` piedras pierde quien mueve cuando `n` es múltiplo de **3** (comprobado en 10.5).

## 9 Minimax con profundidad limitada

La siguiente versión admite:

- una profundidad máxima;
- una función de evaluación para estados no terminales.

Este esquema es mucho más cercano al utilizado en juegos reales.

In [59]:
def minimax_limitado(
    estado,
    profundidad,
    es_max,
    es_terminal,
    utilidad,
    sucesores,
    evaluar
):
    if es_terminal(estado):
        return utilidad(estado)

    if profundidad == 0:
        return evaluar(estado)

    valores = [
        minimax_limitado(
            hijo,
            profundidad - 1,
            not es_max,
            es_terminal,
            utilidad,
            sucesores,
            evaluar
        )
        for hijo in sucesores(estado)
    ]

    return max(valores) if es_max else min(valores)

# 10. Taller

Implemente Minimax para **Tres en raya (Tic-Tac-Toe)**.

Puede representar el tablero como una tupla de nueve posiciones:

```python
(
    "X", "O", " ",
    " ", "X", " ",
    "O", " ", " "
)
```

Suponga:

- `X` es MAX;
- `O` es MIN;
- victoria de `X`: `+1`;
- empate: `0`;
- victoria de `O`: `-1`.

Implemente como mínimo:

```python
acciones(tablero)
resultado(tablero, accion, jugador)
terminal(tablero)
utilidad(tablero)
minimax_tictactoe(tablero, es_max)
```

In [60]:
LINEAS_GANADORAS = [
    (0, 1, 2), (3, 4, 5), (6, 7, 8),   # filas
    (0, 3, 6), (1, 4, 7), (2, 5, 8),   # columnas
    (0, 4, 8), (2, 4, 6),              # diagonales
]

def ganador(tablero):
    """Devuelve 'X', 'O' o None."""
    for a, b, c in LINEAS_GANADORAS:
        if tablero[a] != " " and tablero[a] == tablero[b] == tablero[c]:
            return tablero[a]
    return None

def acciones(tablero):
    """Índices (0-8) de las casillas vacías."""
    return [i for i, casilla in enumerate(tablero) if casilla == " "]

def resultado(tablero, accion, jugador):
    """Nuevo tablero (tupla) tras colocar `jugador` en la casilla `accion`."""
    if tablero[accion] != " ":
        raise ValueError(f"La casilla {accion} ya está ocupada")
    nuevo = list(tablero)
    nuevo[accion] = jugador
    return tuple(nuevo)

def terminal(tablero):
    return ganador(tablero) is not None or " " not in tablero

def utilidad(tablero):
    """+1 si gana X (MAX), -1 si gana O (MIN), 0 en empate."""
    g = ganador(tablero)
    return 1 if g == "X" else -1 if g == "O" else 0

def minimax_tictactoe(tablero, es_max):
    """Valor Minimax del tablero. es_max=True si mueve X."""
    if terminal(tablero):
        return utilidad(tablero)

    jugador = "X" if es_max else "O"
    valores = [
        minimax_tictactoe(resultado(tablero, a, jugador), not es_max)
        for a in acciones(tablero)
    ]
    return max(valores) if es_max else min(valores)

### 10.1 Funciones auxiliares y pruebas con distintos tableros

In [61]:
def quien_mueve(tablero):
    return "X" if tablero.count("X") == tablero.count("O") else "O"

def mostrar_tablero(tablero):
    filas = [" | ".join(tablero[3 * f: 3 * f + 3]) for f in range(3)]
    print("\n---------\n".join(filas))

def mejor_jugada_tictactoe(tablero, es_max):
    """Valor de cada acción y lista de las mejores (puede haber varias)."""
    jugador = "X" if es_max else "O"
    opciones = {
        a: minimax_tictactoe(resultado(tablero, a, jugador), not es_max)
        for a in acciones(tablero)
    }
    mejor = max(opciones.values()) if es_max else min(opciones.values())
    return {
        "valor": mejor,
        "mejores": [a for a, v in opciones.items() if v == mejor],
        "opciones": opciones,
    }

pruebas = {
    "Tablero del enunciado":       ("X", "O", " ", " ", "X", " ", "O", " ", " "),
    "X gana en una jugada":        ("X", "X", " ", "O", "O", " ", " ", " ", " "),
    "O debe bloquear":             ("X", "X", " ", " ", "O", " ", " ", " ", " "),
    "Tablero vacío":               (" ",) * 9,
    "Terminal: gana O":            ("O", "O", "O", "X", "X", " ", "X", " ", " "),
    "Terminal: empate":            ("X", "O", "X", "X", "O", "O", "O", "X", "X"),
}

for nombre, t in pruebas.items():
    print("=" * 50)
    print(nombre)
    mostrar_tablero(t)
    if terminal(t):
        print(f"\nTerminal -> utilidad = {utilidad(t)}")
        continue
    turno = quien_mueve(t)
    r = mejor_jugada_tictactoe(t, turno == "X")
    print(f"\nMueve {turno} | valor Minimax = {r['valor']} | mejores casillas = {r['mejores']}")
    print("Valor de cada casilla:", r["opciones"])

Tablero del enunciado
X | O |  
---------
  | X |  
---------
O |   |  

Mueve X | valor Minimax = 1 | mejores casillas = [3, 5, 8]
Valor de cada casilla: {2: 0, 3: 1, 5: 1, 7: 0, 8: 1}
X gana en una jugada
X | X |  
---------
O | O |  
---------
  |   |  

Mueve X | valor Minimax = 1 | mejores casillas = [2]
Valor de cada casilla: {2: 1, 5: 0, 6: -1, 7: -1, 8: -1}
O debe bloquear
X | X |  
---------
  | O |  
---------
  |   |  

Mueve O | valor Minimax = 0 | mejores casillas = [2]
Valor de cada casilla: {2: 0, 3: 1, 5: 1, 6: 1, 7: 1, 8: 1}
Tablero vacío
  |   |  
---------
  |   |  
---------
  |   |  

Mueve X | valor Minimax = 0 | mejores casillas = [0, 1, 2, 3, 4, 5, 6, 7, 8]
Valor de cada casilla: {0: 0, 1: 0, 2: 0, 3: 0, 4: 0, 5: 0, 6: 0, 7: 0, 8: 0}
Terminal: gana O
O | O | O
---------
X | X |  
---------
X |   |  

Terminal -> utilidad = -1
Terminal: empate
X | O | X
---------
X | O | O
---------
O | X | X

Terminal -> utilidad = 0


**Interpretación (10.1):**

- **Tablero vacío:** valor `0`; con juego óptimo es empate y las nueve aperturas son igual de buenas.
- **Tablero del enunciado** (mueve X): valor `+1`. X gana jugando en 8 (victoria inmediata), pero también en 3 o 5, que obligan a ganar unas jugadas después. Hay varias jugadas igualmente buenas.
- **X gana en una jugada:** la única casilla ganadora es la 2; las demás valen `0` o `-1` (X deja ganar a O).
- **O debe bloquear:** solo la casilla 2 evita la derrota (valor `0`); cualquier otra da `+1` a X.
- Los tableros terminales devuelven directamente su utilidad (`-1` si gana O, `0` si hay empate).

### 10.2 Coste de Minimax y versión con memoización

Minimax visita el mismo tablero muchas veces (se puede llegar a él por varios órdenes de jugadas). Se cuenta el trabajo y se compara con una versión que recuerda los valores ya calculados.

In [62]:
from functools import lru_cache
import time

def minimax_contando(tablero, es_max, contador):
    contador["nodos"] += 1
    if terminal(tablero):
        return utilidad(tablero)
    jugador = "X" if es_max else "O"
    valores = [
        minimax_contando(resultado(tablero, a, jugador), not es_max, contador)
        for a in acciones(tablero)
    ]
    return max(valores) if es_max else min(valores)

@lru_cache(maxsize=None)
def minimax_memo(tablero, es_max):
    if terminal(tablero):
        return utilidad(tablero)
    jugador = "X" if es_max else "O"
    valores = [minimax_memo(resultado(tablero, a, jugador), not es_max) for a in acciones(tablero)]
    return max(valores) if es_max else min(valores)

vacio = (" ",) * 9

contador = {"nodos": 0}
t0 = time.perf_counter()
v1 = minimax_contando(vacio, True, contador)
t1 = time.perf_counter() - t0

t0 = time.perf_counter()
v2 = minimax_memo(vacio, True)
t2 = time.perf_counter() - t0

print(f"Minimax simple  : valor = {v1}, nodos visitados = {contador['nodos']:,}, tiempo = {t1:.2f} s")
print(f"Minimax memoizado: valor = {v2}, tableros distintos = {minimax_memo.cache_info().currsize:,}, tiempo = {t2:.3f} s")

# Las dos versiones coinciden en todos los tableros de prueba
for nombre, t in pruebas.items():
    if not terminal(t):
        es_max = quien_mueve(t) == "X"
        assert minimax_tictactoe(t, es_max) == minimax_memo(t, es_max)
print("Versión simple y memoizada coinciden en todas las pruebas.")

Minimax simple  : valor = 0, nodos visitados = 549,946, tiempo = 2.31 s
Minimax memoizado: valor = 0, tableros distintos = 5,478, tiempo = 0.043 s
Versión simple y memoizada coinciden en todas las pruebas.


**Interpretación (10.2):** desde el tablero vacío Minimax visita **549 946 nodos**, aunque solo existen **5 478 tableros distintos**: un mismo tablero se alcanza por muchos órdenes de jugadas (transposiciones). Memorizar los valores ya calculados da el mismo resultado con una fracción del trabajo. En juegos mayores este crecimiento es lo que motiva la poda Alfa–Beta y la profundidad limitada.

### 10.3 Partidas de comprobación

Un jugador Minimax no debería perder nunca. Se comprueba contra un rival aleatorio (semilla fija) y contra sí mismo.

In [63]:
import random
from collections import Counter

def jugar(estrategia_X, estrategia_O):
    tablero, turno = (" ",) * 9, "X"
    while not terminal(tablero):
        a = (estrategia_X if turno == "X" else estrategia_O)(tablero)
        tablero = resultado(tablero, a, turno)
        turno = "O" if turno == "X" else "X"
    return utilidad(tablero)

def jugador_minimax(marca):
    es_max = marca == "X"
    def elegir(tablero):
        opciones = {a: minimax_memo(resultado(tablero, a, marca), not es_max) for a in acciones(tablero)}
        mejor = max(opciones.values()) if es_max else min(opciones.values())
        return random.choice([a for a, v in opciones.items() if v == mejor])
    return elegir

def jugador_azar(tablero):
    return random.choice(acciones(tablero))

random.seed(42)
N = 300
como_X = Counter(jugar(jugador_minimax("X"), jugador_azar) for _ in range(N))
como_O = Counter(-jugar(jugador_azar, jugador_minimax("O")) for _ in range(N))   # se invierte el signo: +1 = gana Minimax
mm_vs_mm = Counter(jugar(jugador_minimax("X"), jugador_minimax("O")) for _ in range(50))

print(f"Minimax como X vs azar ({N} partidas): gana {como_X[1]}, empata {como_X[0]}, pierde {como_X[-1]}")
print(f"Minimax como O vs azar ({N} partidas): gana {como_O[1]}, empata {como_O[0]}, pierde {como_O[-1]}")
print(f"Minimax vs Minimax (50 partidas)     : X gana {mm_vs_mm[1]}, empates {mm_vs_mm[0]}, O gana {mm_vs_mm[-1]}")

Minimax como X vs azar (300 partidas): gana 287, empata 13, pierde 0
Minimax como O vs azar (300 partidas): gana 230, empata 70, pierde 0
Minimax vs Minimax (50 partidas)     : X gana 0, empates 50, O gana 0


**Interpretación (10.3):** el jugador Minimax **no pierde ninguna partida** (0 derrotas en 300 contra un rival aleatorio jugando como X y 300 jugando como O). Jugando como O gana menos partidas que como X porque mueve segundo. Entre dos jugadores Minimax todas las partidas terminan en empate, consistente con el valor `0` del tablero vacío.

### Para el juego de 3 en raya:

Construya un árbol de al menos tres niveles y:

1. asigne valores de utilidad a las hojas;
2. calcule manualmente los valores Minimax;
3. compruebe el resultado con Python;
4. indique la jugada elegida por MAX.

Complete la implementación propuesta y pruebe diferentes tableros.

### Para el juego de las piedras
Modifique las reglas para permitir retirar únicamente `1`, `2` o `4` piedras.

Analice:

- posiciones ganadoras;
- posiciones perdedoras;
- mejor movimiento para MAX.


### Pregunta final

**¿Por qué una decisión que parece buena de manera inmediata puede ser mala después de considerar la respuesta del adversario?**

### Uso de IA generativa

Si utiliza IA generativa, indique:

- herramienta utilizada;
- propósito de uso;
- partes de la solución en las que fue empleada.

## 10.4 Árbol de tres niveles con Tres en raya

Se parte de un tablero con **tres casillas libres** y turno de X (MAX). Así el árbol tiene tres niveles de decisión: `MAX (X) → MIN (O) → MAX (X)`, y las hojas son tableros terminales.

```
 X | X | O        casillas:  0 | 1 | 2
---+---+---                 ---+---+---
 O |   | X                  3 | 4 | 5
---+---+---                 ---+---+---
 O |   |                    6 | 7 | 8
```

**Paso 1 — utilidades de las hojas.** Se generan todas las jugadas posibles y se asigna a cada hoja su utilidad (`+1` gana X, `-1` gana O, `0` empate).

In [64]:
tablero_arbol = (
    "X", "X", "O",
    "O", " ", "X",
    "O", " ", " ",
)

def construir_arbol(tablero, es_max, nombre="raiz", arbol=None, utilidades=None):
    """Convierte el juego en los diccionarios `arbol` y `utilidades` usados en las secciones 3-7."""
    if arbol is None:
        arbol, utilidades = {}, {}

    if terminal(tablero):
        utilidades[nombre] = utilidad(tablero)
        return arbol, utilidades

    jugador = "X" if es_max else "O"
    hijos = []
    for a in acciones(tablero):
        hijo = f"{nombre}>{jugador}{a}"
        hijos.append(hijo)
        construir_arbol(resultado(tablero, a, jugador), not es_max, hijo, arbol, utilidades)
    arbol[nombre] = hijos
    return arbol, utilidades

arbol_ttt, utilidades_ttt = construir_arbol(tablero_arbol, es_max=True)

def imprimir_arbol(nodo, nivel=0):
    turno = ["MAX (X)", "MIN (O)", "MAX (X)"][nivel] if nodo in arbol_ttt else "hoja"
    extra = f" -> utilidad {utilidades_ttt[nodo]:+d}" if nodo in utilidades_ttt else ""
    print("    " * nivel + f"{nodo.split('>')[-1] if nivel else 'raíz'} [{turno}]{extra}")
    for hijo in arbol_ttt.get(nodo, []):
        imprimir_arbol(hijo, nivel + 1)

imprimir_arbol("raiz")
print("\nHojas:", len(utilidades_ttt), "| utilidades:", utilidades_ttt)

raíz [MAX (X)]
    X4 [MIN (O)]
        O7 [MAX (X)]
            X8 [hoja] -> utilidad +1
        O8 [MAX (X)]
            X7 [hoja] -> utilidad +1
    X7 [MIN (O)]
        O4 [hoja] -> utilidad -1
        O8 [MAX (X)]
            X4 [hoja] -> utilidad +1
    X8 [MIN (O)]
        O4 [hoja] -> utilidad -1
        O7 [MAX (X)]
            X4 [hoja] -> utilidad +1

Hojas: 6 | utilidades: {'raiz>X4>O7>X8': 1, 'raiz>X4>O8>X7': 1, 'raiz>X7>O4': -1, 'raiz>X7>O8>X4': 1, 'raiz>X8>O4': -1, 'raiz>X8>O7>X4': 1}


**Paso 2 — cálculo manual.**

| Rama de MAX | Respuestas de MIN | Valor de las hojas | Valor de MIN |
|---|---|---|---|
| **X4** | O7 → X8 (diagonal 0-4-8) · O8 → X7 (columna 1-4-7) | `+1`, `+1` | `min(+1, +1) = +1` |
| **X7** | O4 (diagonal 2-4-6, gana O) · O8 → X4 (columna 1-4-7) | `-1`, `+1` | `min(-1, +1) = -1` |
| **X8** | O4 (diagonal 2-4-6, gana O) · O7 → X4 (diagonal 0-4-8) | `-1`, `+1` | `min(-1, +1) = -1` |

Valor de la raíz (MAX): `max(+1, -1, -1) = +1` → **MAX debe jugar X4**, es decir, la casilla central.

**Paso 3 — comprobación con Python.** Se usan las funciones `minimax` y `mejor_jugada_minimax` de las secciones 5 y 5.1 sobre el árbol construido, y se contrasta con `minimax_tictactoe`.

In [65]:
valor_arbol = minimax("raiz", True, arbol_ttt, utilidades_ttt)
valor_jugada, hijo = mejor_jugada_minimax("raiz", True, arbol_ttt, utilidades_ttt)
valor_directo = minimax_tictactoe(tablero_arbol, True)

print("Valor con minimax()              :", valor_arbol)
print("Valor con minimax_tictactoe()    :", valor_directo)
print("Jugada elegida por MAX (nodo)    :", hijo)
print("Valor de cada rama de MAX        :",
      {h: minimax(h, False, arbol_ttt, utilidades_ttt) for h in arbol_ttt["raiz"]})

assert valor_arbol == valor_directo == 1
assert hijo == "raiz>X4"
print("\nComprobación correcta: coincide con el cálculo manual.\n")

minimax_debug("raiz", True, arbol_ttt, utilidades_ttt)

Valor con minimax()              : 1
Valor con minimax_tictactoe()    : 1
Jugada elegida por MAX (nodo)    : raiz>X4
Valor de cada rama de MAX        : {'raiz>X4': 1, 'raiz>X7': -1, 'raiz>X8': -1}

Comprobación correcta: coincide con el cálculo manual.

raiz: turno de MAX
    raiz>X4: turno de MIN
        raiz>X4>O7: turno de MAX
            raiz>X4>O7>X8: terminal -> utilidad 1
        raiz>X4>O7: MAX selecciona 1
        raiz>X4>O8: turno de MAX
            raiz>X4>O8>X7: terminal -> utilidad 1
        raiz>X4>O8: MAX selecciona 1
    raiz>X4: MIN selecciona 1
    raiz>X7: turno de MIN
        raiz>X7>O4: terminal -> utilidad -1
        raiz>X7>O8: turno de MAX
            raiz>X7>O8>X4: terminal -> utilidad 1
        raiz>X7>O8: MAX selecciona 1
    raiz>X7: MIN selecciona -1
    raiz>X8: turno de MIN
        raiz>X8>O4: terminal -> utilidad -1
        raiz>X8>O7: turno de MAX
            raiz>X8>O7>X4: terminal -> utilidad 1
        raiz>X8>O7: MAX selecciona 1
    raiz>X8: MIN sel

1

**Paso 4 — jugada elegida:** MAX (X) juega en la **casilla 4** (centro), con valor Minimax `+1`.

## 10.5 Juego de las piedras retirando `1`, `2` o `4`

Primero se modifica la regla en el propio notebook (variable global `MOVIMIENTOS`) y se reutilizan `minimax_piedras` y `mejor_movimiento_piedras`; después se generaliza con una función parametrizada y memoizada para analizar más posiciones.

In [66]:
MOVIMIENTOS_ORIGINALES = MOVIMIENTOS      # (1, 2, 3)
MOVIMIENTOS = (1, 2, 4)                   # nueva regla

for piedras in range(1, 13):
    r = mejor_movimiento_piedras(piedras)
    print(f"{piedras:2d} piedras -> valor = {r['valor']:+d} | opciones (valor, retirar) = {r['opciones']}")

MOVIMIENTOS = MOVIMIENTOS_ORIGINALES      # se restaura la regla original

 1 piedras -> valor = +1 | opciones (valor, retirar) = [(1, 1)]
 2 piedras -> valor = +1 | opciones (valor, retirar) = [(-1, 1), (1, 2)]
 3 piedras -> valor = -1 | opciones (valor, retirar) = [(-1, 1), (-1, 2)]
 4 piedras -> valor = +1 | opciones (valor, retirar) = [(1, 1), (-1, 2), (1, 4)]
 5 piedras -> valor = +1 | opciones (valor, retirar) = [(-1, 1), (1, 2), (-1, 4)]
 6 piedras -> valor = -1 | opciones (valor, retirar) = [(-1, 1), (-1, 2), (-1, 4)]
 7 piedras -> valor = +1 | opciones (valor, retirar) = [(1, 1), (-1, 2), (1, 4)]
 8 piedras -> valor = +1 | opciones (valor, retirar) = [(-1, 1), (1, 2), (-1, 4)]
 9 piedras -> valor = -1 | opciones (valor, retirar) = [(-1, 1), (-1, 2), (-1, 4)]
10 piedras -> valor = +1 | opciones (valor, retirar) = [(1, 1), (-1, 2), (1, 4)]
11 piedras -> valor = +1 | opciones (valor, retirar) = [(-1, 1), (1, 2), (-1, 4)]
12 piedras -> valor = -1 | opciones (valor, retirar) = [(-1, 1), (-1, 2), (-1, 4)]


In [67]:
def analizar_piedras(movs, n_max=30):
    """Para cada n: valor Minimax para MAX (que mueve primero) y movimientos óptimos."""
    @lru_cache(maxsize=None)
    def valor(n, turno_max):
        if n == 0:
            return -1 if turno_max else 1
        vs = [valor(n - m, not turno_max) for m in movs if m <= n]
        return max(vs) if turno_max else min(vs)

    filas = []
    for n in range(1, n_max + 1):
        opciones = {m: valor(n - m, False) for m in movs if m <= n}
        v = max(opciones.values())
        filas.append((n, v, [m for m, x in opciones.items() if x == v]))
    return filas

filas = analizar_piedras((1, 2, 4), 30)

# Verificación cruzada con las funciones originales del notebook
MOVIMIENTOS = (1, 2, 4)
for n, v, mejores in filas[:14]:
    assert minimax_piedras(n, True) == v
    assert mejor_movimiento_piedras(n)["retirar"] in mejores
MOVIMIENTOS = MOVIMIENTOS_ORIGINALES
print("Las funciones originales y la versión parametrizada coinciden (n = 1..14).\n")

print(f"{'n':>3} {'valor':>6} {'tipo':>11}   movimientos óptimos")
for n, v, mejores in filas[:15]:
    tipo = "GANADORA" if v == 1 else "perdedora"
    opt = mejores if v == 1 else "ninguno (todos pierden)"
    print(f"{n:3d} {v:+6d} {tipo:>11}   {opt}")

perdedoras = [n for n, v, _ in filas if v == -1]
print("\nPosiciones perdedoras (n ≤ 30):", perdedoras)
print("¿Son exactamente los múltiplos de 3?", perdedoras == list(range(3, 31, 3)))
dobles = [n for n, v, m in filas if v == 1 and len(m) > 1]
print("Posiciones con más de una jugada ganadora:", dobles)

Las funciones originales y la versión parametrizada coinciden (n = 1..14).

  n  valor        tipo   movimientos óptimos
  1     +1    GANADORA   [1]
  2     +1    GANADORA   [2]
  3     -1   perdedora   ninguno (todos pierden)
  4     +1    GANADORA   [1, 4]
  5     +1    GANADORA   [2]
  6     -1   perdedora   ninguno (todos pierden)
  7     +1    GANADORA   [1, 4]
  8     +1    GANADORA   [2]
  9     -1   perdedora   ninguno (todos pierden)
 10     +1    GANADORA   [1, 4]
 11     +1    GANADORA   [2]
 12     -1   perdedora   ninguno (todos pierden)
 13     +1    GANADORA   [1, 4]
 14     +1    GANADORA   [2]
 15     -1   perdedora   ninguno (todos pierden)

Posiciones perdedoras (n ≤ 30): [3, 6, 9, 12, 15, 18, 21, 24, 27, 30]
¿Son exactamente los múltiplos de 3? True
Posiciones con más de una jugada ganadora: [4, 7, 10, 13, 16, 19, 22, 25, 28]


Comparación con otros conjuntos de movimientos permitidos:

In [68]:
for movs in [(1, 2, 3), (1, 2), (1, 2, 4)]:
    perd = [n for n, v, _ in analizar_piedras(movs, 30) if v == -1]
    print(f"Movimientos {movs}: posiciones perdedoras (n ≤ 30) = {perd}")

Movimientos (1, 2, 3): posiciones perdedoras (n ≤ 30) = [4, 8, 12, 16, 20, 24, 28]
Movimientos (1, 2): posiciones perdedoras (n ≤ 30) = [3, 6, 9, 12, 15, 18, 21, 24, 27, 30]
Movimientos (1, 2, 4): posiciones perdedoras (n ≤ 30) = [3, 6, 9, 12, 15, 18, 21, 24, 27, 30]


### Análisis (piedras con `1, 2, 4`)

- **Posiciones perdedoras** (para quien mueve): `3, 6, 9, …`, es decir, los **múltiplos de 3**.
- **Posiciones ganadoras:** todas las demás (`n mod 3 = 1` o `2`).
- **Mejor movimiento de MAX:** si `n mod 3 = 2`, retirar **2**; si `n mod 3 = 1`, retirar **1** (o **4** cuando `n ≥ 4`; ambas son igual de buenas: `4, 7, 10, …`). Si `n` es múltiplo de 3, MAX pierde con juego óptimo del rival.
- **Por qué:** ninguno de `1, 2, 4` es múltiplo de 3, así que desde un múltiplo de 3 cualquier jugada deja un no múltiplo; y desde un no múltiplo siempre existe una jugada que deja un múltiplo de 3.
- **Comparación:** con `1, 2, 3` las perdedoras son los múltiplos de 4; con `1, 2` y con `1, 2, 4` son los múltiplos de 3. Permitir retirar 4 no cambia el patrón de `1, 2` (`4 ≡ 1 mod 3` no añade ningún resto nuevo), solo añade una segunda jugada ganadora.

### Respuesta a la pregunta final

**¿Por qué una decisión que parece buena de forma inmediata puede ser mala al considerar la respuesta del adversario?**

Porque una jugada no se juzga por lo que consigue en ese instante, sino por el valor de la posición **después de la mejor réplica del rival**. Ejemplos de este notebook:

- **Tres en raya (10.4):** X7 y X8 crean una amenaza inmediata (columna 1-4-7 o diagonal 0-4-8), pero dejan libre el centro y O gana con la diagonal 2-4-6 (valor `-1`). X4 ocupa la casilla clave y genera dos amenazas a la vez, que O no puede bloquear (valor `+1`).
- **Piedras con `1, 2, 3`:** con 5 piedras, retirar 3 parece avanzar mucho (quedan 2), pero el rival retira 2 y gana. Lo correcto es retirar 1 y dejar 4, un múltiplo de 4.

Minimax formaliza esto: propaga los valores desde las hojas suponiendo que el adversario responde de la mejor forma posible.

### Uso de IA generativa

- **Herramienta utilizada:** Claude (Anthropic), Gemini (Google).
* **Propósito de uso:** apoyo como herramienta complementaria durante el desarrollo del taller, principalmente para orientar la elaboración de la solución, revisar y validar las implementaciones, apoyar la ejecución e interpretación de pruebas y contribuir a la redacción y revisión de las respuestas de análisis.
- **Partes en las que se empleó:** implementación de Tres en raya (sección 10 y 10.1–10.3), árbol de tres niveles (10.4), juego de las piedras con `1, 2, 4` (10.5) y respuestas a las preguntas de análisis de las secciones 6 y 8.
* **Participación del estudiante:** las decisiones sobre la solución, adaptación del código, ejecución y verificación de las pruebas, interpretación de los resultados y elaboración final del trabajo fueron realizadas y revisadas por el estudiante, utilizando la IA como herramienta de apoyo.